# E05 · Unscented transform and particle filters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/06_estimation_perception/E05_unscented_transform_and_particle_filters.ipynb)

**Track:** Estimation and perception · **Skill axis (sheet):** Estimation and perception · **Time:** about 60 min

**Skill Hub concepts:** Unscented Kalman filter; Particle filter

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("E05")

## Map: two ways around linearization

**Unscented transform (the core of the UKF):** instead of a Jacobian, push $2n+1$ carefully chosen **sigma points**
through the nonlinear function and refit a Gaussian.
$$\lambda = \alpha^2(n + \kappa) - n, \quad \mathcal{X}_0 = \mu, \quad \mathcal{X}_{\pm i} = \mu \pm \big[\sqrt{(n+\lambda)P}\,\big]_i$$
$$W^m_0 = \frac{\lambda}{n+\lambda}, \quad W^c_0 = W^m_0 + (1 - \alpha^2 + \beta), \quad W_i = \frac{1}{2(n+\lambda)}$$
$$\mu_y = \sum W^m_i\, f(\mathcal{X}_i), \qquad P_y = \sum W^c_i\,(f(\mathcal{X}_i) - \mu_y)(f(\mathcal{X}_i) - \mu_y)^\top$$
Typical: $\alpha = 10^{-3}$ to $1$, $\beta = 2$ (Gaussian), $\kappa = 0$. Use the Cholesky factor for the square root.

**Particle filter:** represent the belief by samples, so it can be **multimodal** (the robot could be at either door).
```
 predict: move every particle with the motion model plus noise
 weight:  w_i ∝ p(z | x_i)         normalize
 resample when the effective sample size N_eff = 1 / Σ w_i^2 drops (systematic resampling: one random offset, N evenly spaced pointers)
```

In [ ]:
import numpy as np

## Task 1: sigma points and the unscented transform

In [ ]:
def sigma_points(mu, P, alpha=1.0, beta=2.0, kappa=0.0):
    """Return (X, Wm, Wc) with X of shape (2n+1, n)."""
    raise NotImplementedError  # TODO

def unscented_transform(f, mu, P, **kw):
    raise NotImplementedError  # TODO

lab.check("unscented", sigma_points, unscented_transform)

## Task 2: localize a robot in a corridor with a door sensor
Doors at 2, 6 and 13 m. The sensor says 1 near a door (within 0.5 m) with probability 0.9, and 1 elsewhere with
probability 0.1. The checker drives the robot 112 steps of +0.125 m (your motion model uses noise sigma 0.03 m)
and expects the cloud to collapse onto the true position. Small steps mean several readings per door, which is what
breaks the ambiguity: with 0.5 m steps the same filter converges only about a third of the time.

In [ ]:
DOORS = np.array([2.0, 6.0, 13.0])

def pf_predict(particles, u, sigma, rng):
    raise NotImplementedError  # TODO

def pf_weights(particles, z):
    raise NotImplementedError  # TODO

def systematic_resample(particles, w, rng):
    raise NotImplementedError  # TODO

def n_eff(w):
    raise NotImplementedError  # TODO

lab.check("particles", pf_predict, pf_weights, systematic_resample, n_eff)

## Checkpoint: Math
**M1 (Apply).** Sigma points for a 4-dimensional state?

**M2 (Apply).** Effective sample size of weights [0.5, 0.5, 0, 0]?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Analyze).** The robot starts with no idea where it is and sees its first door. What does the particle cloud look like?
- a) one tight cluster
- b) three clusters, one at each door; only further motion and readings decide which is right
- c) uniform

**K2 (Evaluate).** Compare EKF, UKF and particle filter for (1) a joint velocity estimator and (2) global localization
of a mobile robot in a building.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `e05.h`).
3. In `roboprep/checks/e05.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()